# OpenAI Function Calling Deep Dive

This notebook explores how to implement function calling with OpenAI models. We will use real customer and order data to build a tool that can look up order status and customer details.

In [17]:
import os
import json
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Load real data for grounded examples
customers_df = pd.read_csv('customers.csv')
orders_df = pd.read_csv('order_mock_data.csv')

print("Data loaded successfully.")

Data loaded successfully.


## 1. Defining the Tools

We define functions that the LLM can request to call. These functions will act as the "bridge" to our CSV data.

In [18]:
def get_customer_info(customer_id: str = None, name: str = None):
    """Get customer details by customer_id OR by name.
    Orders return a customer name (not an ID), so this tool accepts either."""
    if customer_id:
        match = customers_df[customers_df['customer_id'] == customer_id]
    elif name:
        match = customers_df[customers_df['name'].str.lower() == name.lower()]
    else:
        return {"error": "Provide customer_id or name"}
    if not match.empty:
        return match.to_dict('records')[0]
    return {"error": "Customer not found"}

In [19]:
def get_order_status(order_id):
    """Get the current status of an order using the order ID."""
    # Normalize: CSV stores IDs as '#ORDNO12', LLM may pass 'ORDNO12' or '#ORDNO12'
    if not order_id.startswith("#"):
        order_id = "#" + order_id
    order = orders_df[orders_df['id'] == order_id]
    if not order.empty:
        return order.to_dict('records')[0]
    return {"error": "Order not found"}

In [20]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_customer_info",
            "description": "Get customer details (tier, email, total spent). Accepts customer_id ('CUST005') OR full name ('Mark Perez'). Use 'name' when chaining from get_order_status, since orders only return the customer's name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string", "description": "Customer ID like 'CUST005' (optional)."},
                    "name": {"type": "string", "description": "Full customer name like 'Mark Perez' (optional)."}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": "Get status, customer name, invoice, and items for an order. Order IDs look like '#ORDNO12'. Returns the customer's NAME, not their customer_id — pass that name to get_customer_info for tier/email.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The order ID in the form '#ORDNO12' (leading '#' optional)."}
                },
                "required": ["order_id"]
            }
        }
    }
]

## 2. The Function Calling Loop

The process follows these steps:
1. Send prompt and tool definitions to OpenAI.
2. OpenAI decides which tool to call and provides arguments.
3. Execute the local Python function with those arguments.
4. Send the tool result back to OpenAI for a final natural language response.

In [21]:
MODEL = "gpt-4o-mini"

available_functions = {
    "get_customer_info": get_customer_info,
    "get_order_status": get_order_status,
}

def run_conversation(user_prompt, max_iters=6):
    """Multi-turn loop: keeps calling the model until it stops requesting tools."""
    messages = [{"role": "user", "content": user_prompt}]

    for _ in range(max_iters):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls

        if not tool_calls:
            return response_message.content

        messages.append(response_message)

        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            print(f"-> Calling {function_name}({function_args})")
            result = available_functions[function_name](**function_args)
            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": function_name,
                "content": json.dumps(result),
            })

    return "[stopped: hit max_iters]"

In [22]:
query = "Can you tell me the status of order #ORDNO12 and who the customer is?"
print(f"Query: {query}\n")
print(f"Response: {run_conversation(query)}")

Query: Can you tell me the status of order #ORDNO12 and who the customer is?

-> Calling get_order_status({'order_id': '#ORDNO12'})
Response: The status of order #ORDNO12 is "Out for Delivery." The customer is Mark Perez. The invoice number for this order is INV-906044, and the items ordered include:

- Ergonomic Chair
- Webcam 1080p
- Mechanical Keyboard


## 3. Building the Same Agent with LangChain `create_agent`

The loop above is educational, but in production you'd use a framework. `create_react_agent` from `langgraph.prebuilt` (the current LangChain agent entry point) wraps the same pattern — tool binding, tool execution, message history — into one call.

Same OpenAI key from `.env`, same two tools, no manual loop.

In [23]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

@tool
def lc_get_customer_info(customer_id: str = "", name: str = "") -> dict:
    """Get customer details (tier, email, total spent).
    Accepts customer_id ('CUST005') OR full name ('Mark Perez').
    When chaining from lc_get_order_status, pass the returned name here."""
    return get_customer_info(
        customer_id=customer_id or None,
        name=name or None,
    )

@tool
def lc_get_order_status(order_id: str) -> dict:
    """Get status, customer name, invoice, and items for an order.
    IDs look like '#ORDNO12'. Returns the customer's NAME (not customer_id)."""
    return get_order_status(order_id)

llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))
agent = create_agent(llm, tools=[lc_get_customer_info, lc_get_order_status])

result = agent.invoke({
    "messages": [("user", "Who placed order #ORDNO12 and what tier are they?")]
})
print(result["messages"][-1].content)

Mark Perez placed order #ORDNO12, and he is in the Gold tier.
